In [3]:
import os
import glob
import pandas as pd

In [4]:
import os
import re

# Folder where your files are stored
folder = '/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data'

# Regex pattern:
# ^\d+_  → matches digits at the beginning followed by underscore
pattern = re.compile(r"^\d+_")

for filename in os.listdir(folder):
    old_path = os.path.join(folder, filename)

    # Skip folders
    if not os.path.isfile(old_path):
        continue

    # Apply renaming only if pattern matches
    if pattern.match(filename):
        new_filename = pattern.sub("", filename)  # Remove leading digits + underscore
        new_path = os.path.join(folder, new_filename)
        
        print(f"Renaming: {filename} → {new_filename}")
        os.rename(old_path, new_path)

print("Done!")


Done!


In [5]:
import pandas as pd
import os

def find_column(df, keywords):
    for col in df.columns:
        name = col.lower()
        if all(k.lower() in name for k in keywords):
            return col
    return None


def process_gsr_data_fullsegment(file_path, output_csv_name):
    print(f"Processing file: {file_path}...")

    # Detect header row (skip metadata)
    header_row_index = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if line.startswith("Row,Timestamp"):
                header_row_index = i
                break

    df = pd.read_csv(file_path, skiprows=header_row_index)
    df.columns = df.columns.str.strip()

    # Find GSR columns
    col_cond = find_column(df, ["conductance"])
    col_tonic = find_column(df, ["tonic"])
    col_phasic = find_column(df, ["phasic"])

    if not col_cond or not col_tonic or not col_phasic:
        print("Error: Missing GSR columns")
        print("Available:", df.columns.tolist())
        return

    # Keep relevant columns
    df = df[['Timestamp', 'SlideEvent', 'SourceStimuliName',
             col_cond, col_tonic, col_phasic]].copy()

    df.rename(columns={
        col_cond: 'GSR_Conductance',
        col_tonic: 'GSR_Tonic',
        col_phasic: 'GSR_Phasic'
    }, inplace=True)

    # Convert to numeric
    df['GSR_Conductance'] = pd.to_numeric(df['GSR_Conductance'], errors='coerce')
    df['GSR_Tonic'] = pd.to_numeric(df['GSR_Tonic'], errors='coerce')
    df['GSR_Phasic'] = pd.to_numeric(df['GSR_Phasic'], errors='coerce')

    # Locate start and end points
    start_indices = df.index[df['SlideEvent'] == 'StartMedia'].tolist()
    end_indices = df.index[df['SlideEvent'] == 'EndMedia'].tolist()

    if not start_indices or not end_indices:
        print("Error: Cannot find both StartMedia and EndMedia")
        return

    start_idx = start_indices[0]
    valid_end_indices = [i for i in end_indices if i > start_idx]

    if not valid_end_indices:
        print("Error: No EndMedia after StartMedia")
        return

    end_idx = valid_end_indices[0]

    # Extract entire segment
    segment_df = df.loc[start_idx:end_idx].copy()

    # Add time elapsed
    start_time = segment_df['Timestamp'].iloc[0]
    segment_df['TimeElapsed'] = segment_df['Timestamp'] - start_time

    # Add metadata
    segment_df['Start_Index'] = start_idx
    segment_df['End_Index'] = end_idx
    segment_df['Start_Timestamp'] = segment_df['Timestamp'].iloc[0]
    segment_df['End_Timestamp'] = segment_df['Timestamp'].iloc[-1]

    # Save full dataset segment
    segment_df.to_csv(output_csv_name, index=False)

    print(f"✔ Full segment saved to {output_csv_name}")
    print(f"Rows exported: {len(segment_df)}")
    print(f"Start index: {start_idx} | End index: {end_idx}")


In [10]:
import os
import glob

# --- Batch processing across participants ---

import os

raw_folder = "/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data"
result_folder = "/Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR"

os.makedirs(result_folder, exist_ok=True)

for filename in os.listdir(raw_folder):
    file_path = os.path.join(raw_folder, filename)

    # 🔥 MUST SKIP DIRECTORIES FIRST
    if os.path.isdir(file_path):
        print(f"Skipping directory: {filename}")
        continue

    # 🔥 Only process CSV files
    if not filename.lower().endswith(".csv"):
        print(f"Skipping non-CSV file: {filename}")
        continue

    # OUTPUT FILE NAME
    output_name = filename.replace(".csv", "_processed.csv")
    output_path = os.path.join(result_folder, output_name)

    print(f"\n=== Running {filename} ===")
    process_gsr_data_fullsegment(file_path, output_path)


=== Running sub008_11-11-25 16h43m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data/sub008_11-11-25 16h43m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53216/3781438020.py:23: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR/sub008_11-11-25 16h43m_processed.csv
Rows exported: 41033
Start index: 5 | End index: 41037

=== Running sub026_13-11-25 09h21m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data/sub026_13-11-25 09h21m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53216/3781438020.py:23: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR/sub026_13-11-25 09h21m_processed.csv
Rows exported: 41000
Start index: 4 | End index: 41003

=== Running sub063_17-11-25 16h18m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data/sub063_17-11-25 16h18m.csv...


...

✔ Full segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR/sub156_02-12-25 15h47m_processed.csv
Rows exported: 40894
Start index: 9 | End index: 40902

=== Running sub187_04-12-25 12h59m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data/sub187_04-12-25 12h59m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53216/3781438020.py:23: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR/sub187_04-12-25 12h59m_processed.csv
Rows exported: 40895
Start index: 7 | End index: 40901

=== Running sub083_24-11-25 10h27m.csv ===
Processing file: /Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/00_GSR_data/sub083_24-11-25 10h27m.csv...


/var/folders/rd/1pmt50jx3xn6cc_31qb4zrn00000gn/T/ipykernel_53216/3781438020.py:23: DtypeWarning: Columns (3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=header_row_index)


✔ Full segment saved to /Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/00_processed_GSR/sub083_24-11-25 10h27m_processed.csv
Rows exported: 40892
Start index: 7 | End index: 40898
